In [1]:
import json
import pandas as pd

# The notebook runs from notebooks/, so the data is one folder up
df = pd.read_parquet("../data/final/site_week_table_fe.parquet")

with open("../data/final/feature_sets.json") as f:
    feature_sets = json.load(f)

print("Rows, columns:", df.shape)
print("Feature set names:", list(feature_sets.keys()))


Rows, columns: (21718, 46)
Feature set names: ['labels', 'split_column', 'regional_core', 'regional_candidates', 'site_core', 'site_candidates', 'do_not_use']


In [2]:
# y_class is missing when the next week wasn't sampled, so those rows can't be used for training or scoring
df = df[df["y_class"].notna()].copy()
df["y_class"] = df["y_class"].astype(int)

print("Rows with a label:", len(df))


Rows with a label: 16037


In [3]:
# Rows = train / validate / test / holdout, columns = class 0 / 1 / 2
print(pd.crosstab(df["split"], df["y_class"], margins=True))


y_class       0     1     2    All
split                             
holdout     550     0     0    550
test       1378    49    28   1455
train      9783  1229  1367  12379
validate   1485   123    45   1653
All       13196  1401  1440  16037


In [4]:
features = feature_sets["site_core"]

print("Number of features:", len(features))
print(features)

# How many values are missing in each feature (largest first)
print(df[features].isna().sum().sort_values(ascending=False))


Number of features: 12
['site_last_class', 'site_last_log_cells', 'site_weeks_since_obs', 'nbr_class', 'reg_last_class', 'reg_class_max_4wk', 'reg_n_samples', 'weeks_since_bloom', 'pos_weeks_12wk', 'class_run_weeks', 'season_sin', 'season_cos']
weeks_since_bloom       1111
nbr_class                268
site_last_class          111
site_last_log_cells      111
site_weeks_since_obs     111
reg_class_max_4wk         14
reg_last_class             0
reg_n_samples              0
pos_weeks_12wk             0
class_run_weeks            0
season_sin                 0
season_cos                 0
dtype: int64


In [5]:
import sklearn
print(sklearn.__version__)


1.9.1


In [6]:
from sklearn.metrics import f1_score, balanced_accuracy_score, recall_score, confusion_matrix

# A reusable "report card" so every model is graded the same way
def evaluate(name, y_true, y_pred, last_class):
    print(f"===== {name} =====")
    print("Macro-F1:          ", round(f1_score(y_true, y_pred, average="macro"), 3))
    print("Balanced accuracy: ", round(balanced_accuracy_score(y_true, y_pred), 3))

    # Recall = out of all the real weeks of each class, how many did we get right?
    recalls = recall_score(y_true, y_pred, average=None, labels=[0, 1, 2])
    print("Recall class 0 / 1 / 2:", recalls.round(3))

    # Onset = the site had no bloom last time, but next week it does (a NEW bloom)
    onset = (last_class == 0) & (y_true > 0)
    caught = (y_pred[onset] > 0).mean()
    print(f"Onset recall:       {caught:.3f}  (out of {onset.sum()} new blooms)")

    # Confusion matrix: rows = what really happened, columns = what we predicted
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
    print(pd.DataFrame(cm, index=["true 0", "true 1", "true 2"],
                       columns=["pred 0", "pred 1", "pred 2"]))
    print()


Split Data by Year

In [7]:
# Learn from the older years, check on 2024 (don't touch 2025 yet)
train = df[df["split"] == "train"]
val = df[df["split"] == "validate"]

X_train, y_train = train[features], train["y_class"]   # X = features (clues), y = answer
X_val, y_val = val[features], val["y_class"]

print("Train rows:", len(train), "| Validate rows:", len(val))


Train rows: 12379 | Validate rows: 1653


 Baseline, "next week = last week"

In [8]:
# No learning: just copy the site's last class. If there's no last class, guess 0 (no bloom)
baseline_pred = val["site_last_class"].fillna(0).astype(int)

evaluate("Baseline: next week = last week", y_val, baseline_pred, val["site_last_class"])


===== Baseline: next week = last week =====


Macro-F1:           0.49
Balanced accuracy:  0.478
Recall class 0 / 1 / 2: [0.957 0.301 0.178]
Onset recall:       0.000  (out of 90 new blooms)
        pred 0  pred 1  pred 2
true 0    1421      51      13
true 1      71      37      15
true 2      19      18       8



Train random Forest

In [9]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=500,          # 500 trees that vote
    class_weight="balanced",   # count the rare bloom weeks more heavily
    min_samples_leaf=10,       # each tree "rule" must cover at least 10 rows (prevents memorizing)
    random_state=42,           # same result every time you run it
    n_jobs=-1,                 # use all your computer's cores to go faster
)

rf.fit(X_train, y_train)       # learn from the training years
rf_pred = rf.predict(X_val)    # predict 2024

evaluate("Random forest (12 core features)", y_val, rf_pred, val["site_last_class"])


===== Random forest (12 core features) =====
Macro-F1:           0.472
Balanced accuracy:  0.499
Recall class 0 / 1 / 2: [0.91  0.341 0.244]
Onset recall:       0.511  (out of 90 new blooms)
        pred 0  pred 1  pred 2
true 0    1352      45      88
true 1      30      42      51
true 2      14      20      11



In [10]:
from sklearn.inspection import permutation_importance

# Shuffle each feature 10 times on the 2024 data and measure how much macro-F1 drops
result = permutation_importance(
    rf, X_val, y_val,
    scoring="f1_macro",    # the same score we used for the report card
    n_repeats=10,          # shuffle each feature 10 times and average
    random_state=42,       # same result every run
    n_jobs=-1,             # use all cores
)

# Put the results in a table, most important first
importance = pd.DataFrame({
    "feature": features,
    "importance": result.importances_mean,   # average drop in score
    "spread": result.importances_std,        # how much the 10 shuffles disagreed
}).sort_values("importance", ascending=False)

print(importance.round(4).to_string(index=False))


             feature  importance  spread
      pos_weeks_12wk      0.0328  0.0052
          season_cos      0.0261  0.0088
     site_last_class      0.0085  0.0023
 site_last_log_cells      0.0057  0.0082
   reg_class_max_4wk      0.0009  0.0026
     class_run_weeks     -0.0043  0.0061
           nbr_class     -0.0052  0.0090
       reg_n_samples     -0.0093  0.0088
site_weeks_since_obs     -0.0098  0.0079
      reg_last_class     -0.0102  0.0051
          season_sin     -0.0168  0.0157
   weeks_since_bloom     -0.0253  0.0069


# Part 2: Adjusting the model to the client's feedback

The client asked for:

1. **Onset recall as a primary metric**, reported alongside bloom misses and false alarms
2. **A regional next-week forecast as the main product**; site-level is secondary until it is shown to be reliable across sites
3. **Misses cost more than false alarms**: start with 2:1 or 3:1, and still show the raw counts
4. **Keep environmental features only if they help validation**, and document what was tested

The forecast is now an **alert**: "next week will be at risk or bloom (5,000+ cells/L)". The forest still learns all three classes; an alert is raised when its probability of class 1 or 2 is above a cutoff (the threshold).

## Early-warning report card

In [11]:
import numpy as np
pd.set_option("display.max_columns", None)   # show every metric column

# Cost of a missed bloom week compared with a false alarm. 3:1 is the main one; 2:1 is reported too
MAIN_RATIO = 3

from sklearn.metrics import accuracy_score

def early_warning_report(y_true, pred_class, last_class):
    y_true = np.asarray(y_true)
    pred_class = np.asarray(pred_class)
    last_class = np.asarray(last_class)

    alert = pred_class > 0                          # predicted at risk or bloom
    risky = y_true > 0                              # really at risk or bloom next week
    onset = (last_class == 0) & risky               # a NEW bloom: no bloom last time, risk/bloom next week
    misses = int((risky & ~alert).sum())            # risky week with no alert
    false_alarms = int((~risky & alert).sum())      # alert in a week that turned out fine

    return {
        # Overall scores on the three classes (same as Part 1)
        "accuracy": round(accuracy_score(y_true, pred_class), 3),
        "balanced_accuracy": round(balanced_accuracy_score(y_true, pred_class), 3),
        "macro_f1": round(f1_score(y_true, pred_class, average="macro", labels=[0, 1, 2], zero_division=0), 3),
        # Early-warning scores
        "onset_recall": round(alert[onset].mean(), 3) if onset.sum() else np.nan,
        "onsets_caught": f"{int(alert[onset].sum())} / {int(onset.sum())}",
        "misses": misses,
        "bloom_misses": int(((y_true == 2) & ~alert).sum()),   # the worst misses: real bloom (class 2), no alert
        "false_alarms": false_alarms,
        "cost_2to1": 2 * misses + false_alarms,
        "cost_3to1": 3 * misses + false_alarms,
    }

### Rescoring the Part 1 models (2024)

Same baseline and forest as above. The forest's alert is "predicted class 1 or 2".

In [12]:
rescored = pd.DataFrame({
    "Baseline: next week = last week": early_warning_report(y_val, baseline_pred, val["site_last_class"]),
    "Random forest (Part 1)": early_warning_report(y_val, rf_pred, val["site_last_class"]),
}).T
rescored

,accuracy,balanced_accuracy,macro_f1,onset_recall,onsets_caught,misses,bloom_misses,false_alarms,cost_2to1,cost_3to1
Baseline: next week = last week,0.887,0.478,0.49,0.0,0 / 90,90,19,64,244,334
Random forest (Part 1),0.85,0.499,0.472,0.511,46 / 90,44,14,133,221,265


## Rolling validation

2024 alone is a small test (and at the regional level it is only 52 weeks), so choices made on it can be luck. Instead, each validation year is predicted by a model trained only on the years before it, and the results are added together:

| Validate on | Train on |
| ----------- | -------- |
| 2019 | 2015–2018 |
| 2020 | 2015–2019 |
| 2021 | 2015–2020 |
| 2024 | 2015–2023 |

2022 and 2023 are skipped as validation years: they have almost no labels (15 and 48 site-weeks, 8 and 17 regional weeks). They are still used for training. 2025 stays untouched until the very end.

In [13]:
VAL_YEARS = [2019, 2020, 2021, 2024]
THRESHOLDS = np.round(np.arange(0.05, 0.96, 0.05), 2)

def make_forest(class_weight="balanced", leaf=30, seed=42):
    return RandomForestClassifier(n_estimators=500, class_weight=class_weight, min_samples_leaf=leaf,
                                  random_state=seed, n_jobs=-1)

def rolling_predictions(data, feats, last_col, val_years=VAL_YEARS, start_year=None, **forest_args):
    """Predict each validation year with a forest trained only on earlier years."""
    pieces = []
    for year in val_years:
        tr = data[data["target_year"] < year]
        if start_year:
            tr = tr[tr["target_year"] >= start_year]   # e.g. only years where chlorophyll exists
        va = data[data["target_year"] == year]

        forest = make_forest(**forest_args).fit(tr[feats], tr["y_class"])
        proba = forest.predict_proba(va[feats])

        keep = [c for c in ["site_key", "week_start", "target_year", "latitude", "inshore", "y_class", last_col] if c in va]
        out = va[keep].copy()
        out["p_alert"] = 1 - proba[:, list(forest.classes_).index(0)]   # probability of class 1 or 2
        out["p_bloom"] = proba[:, list(forest.classes_).index(2)]       # probability of class 2
        pieces.append(out)
    return pd.concat(pieces)

def to_class(preds, threshold):
    """No alert -> 0. Alert -> 2 if bloom is more likely than at-risk, otherwise 1."""
    p_risk = preds["p_alert"] - preds["p_bloom"]
    severity = np.where(preds["p_bloom"] >= p_risk, 2, 1)
    return np.where(preds["p_alert"] >= threshold, severity, 0)

def threshold_sweep(preds, last_col):
    rows = []
    for t in THRESHOLDS:
        row = early_warning_report(preds["y_class"], to_class(preds, t), preds[last_col])
        rows.append({"threshold": t, **row})
    return pd.DataFrame(rows)

def pick_threshold(preds, last_col, ratio=MAIN_RATIO, tolerance=0.05):
    """Lowest cost wins. If other thresholds are within 5% of that cost, take the one that catches the most onsets."""
    sweep = threshold_sweep(preds, last_col)
    cost = sweep[f"cost_{ratio}to1"]
    close = sweep[cost <= cost.min() * (1 + tolerance)]
    return close.sort_values(["onset_recall", f"cost_{ratio}to1"], ascending=[False, True]).iloc[0]

def best_cost(preds, last_col, ratio=MAIN_RATIO):
    """The lowest cost any threshold reaches. Used to compare models and features fairly."""
    return threshold_sweep(preds, last_col)[f"cost_{ratio}to1"].min()

# Site-level model (secondary)

### Baseline and forest settings

Two settings are compared: class weights (`"balanced"` vs none) and leaf size (10 vs 30). Each version gets its own best threshold.

In [14]:
site_val = df[df["target_year"].isin(VAL_YEARS)]
site_baseline = early_warning_report(site_val["y_class"], site_val["site_last_class"].fillna(0).astype(int), site_val["site_last_class"])

rows = [{"model": "Baseline: next week = last week", "threshold": None, **site_baseline}]
for cw in ["balanced", None]:
    for leaf in [10, 30]:
        preds = rolling_predictions(df, features, "site_last_class", class_weight=cw, leaf=leaf)
        best = pick_threshold(preds, "site_last_class")
        rows.append({"model": f"Forest, class_weight={cw}, leaf={leaf}", **best.to_dict()})

site_settings = pd.DataFrame(rows)
site_settings

,model,threshold,accuracy,balanced_accuracy,macro_f1,onset_recall,onsets_caught,misses,bloom_misses,false_alarms,cost_2to1,cost_3to1
0,Baseline: next week = last week,NaN,0.896,0.524,0.518,0.000,0 / 291,291,74,302,884,1175
1,"Forest, class_weight=balanced, leaf=10",0.5,0.871,0.621,0.530,0.553,161 / 291,138,31,546,822,960
2,"Forest, class_weight=balanced, leaf=30",0.6,0.874,0.619,0.532,0.519,151 / 291,151,33,520,822,973
3,"Forest, class_weight=None, leaf=10",0.2,0.859,0.602,0.495,0.512,149 / 291,150,33,587,887,1037
4,"Forest, class_weight=None, leaf=30",0.2,0.863,0.600,0.495,0.522,152 / 291,146,32,561,853,999


### How much do results move by chance?

Rerunning the chosen forest with different random seeds shows how big a difference has to be before it means anything. A change only counts as real if it moves the lowest cost by more than **twice** the seed-to-seed range (`SITE_NOISE`).

In [15]:
SITE_ARGS = dict(class_weight="balanced", leaf=30)

seed_costs = [best_cost(rolling_predictions(df, features, "site_last_class", seed=seed, **SITE_ARGS), "site_last_class")
              for seed in [1, 2, 3, 42]]
SITE_NOISE = 2 * (max(seed_costs) - min(seed_costs))

print("Lowest 3:1 cost by seed:", seed_costs)
print("A change has to beat this to count:", SITE_NOISE)

Lowest 3:1 cost by seed: [np.int64(923), np.int64(909), np.int64(921), np.int64(934)]
A change has to beat this to count: 50


### The trade-off at each threshold

This table is the main thing to show the client: a lower threshold catches more onsets but raises more false alarms.

In [16]:
site_preds = rolling_predictions(df, features, "site_last_class", **SITE_ARGS)
site_choice = pick_threshold(site_preds, "site_last_class")
SITE_THRESHOLD = site_choice["threshold"]

print("Chosen threshold:", SITE_THRESHOLD)
threshold_sweep(site_preds, "site_last_class")

Chosen threshold: 0.6


,threshold,accuracy,balanced_accuracy,macro_f1,onset_recall,onsets_caught,misses,bloom_misses,false_alarms,cost_2to1,cost_3to1
0,0.05,0.220,0.486,0.250,0.990,288 / 291,3,0,4848,4854,4857
1,0.10,0.317,0.516,0.300,0.969,282 / 291,9,1,4220,4238,4247
2,0.15,0.451,0.558,0.359,0.942,274 / 291,17,1,3345,3379,3396
3,0.20,0.632,0.615,0.431,0.911,265 / 291,26,1,2172,2224,2250
4,0.25,0.693,0.631,0.455,0.876,255 / 291,36,4,1774,1846,1882
5,0.30,0.751,0.641,0.479,0.825,240 / 291,51,7,1388,1490,1541
6,0.35,0.792,0.648,0.499,0.790,230 / 291,66,13,1109,1241,1307
7,0.40,0.830,0.650,0.519,0.722,210 / 291,86,20,855,1027,1113
8,0.45,0.843,0.649,0.526,0.691,201 / 291,95,23,764,954,1049
9,0.50,0.853,0.640,0.529,0.639,186 / 291,111,25,684,906,1017


## Feature tests (site level)

Each test is compared with the core-feature forest **trained on the same years**:

- Drops: `weeks_since_bloom` and `reg_last_class` hurt in Part 1's permutation importance. Dropping `site_last_class` shows how much skill is just "copy last week".
- Environmental groups are added one at a time. Chlorophyll only exists from 2018 and wind from 2019, so for those, **both** the core and the core-plus-group forests train only on years with that data. Otherwise the test would measure "less training data", not the feature.

Each version is compared on its **lowest** 3:1 cost (`best_cost`). A test **helps** only if it lowers that cost by more than `SITE_NOISE`.

In [17]:
def feature_test(data, core, last_col, name, feats, start_year=None, noise=0, **forest_args):
    years = [y for y in VAL_YEARS if start_year is None or y > start_year]
    base = rolling_predictions(data, core, last_col, years, start_year, **forest_args)
    test = rolling_predictions(data, feats, last_col, years, start_year, **forest_args)
    change = best_cost(test, last_col) - best_cost(base, last_col)   # negative = better
    base_pick, test_pick = pick_threshold(base, last_col), pick_threshold(test, last_col)
    return {
        "test": name,
        "trained_from": start_year or 2015,
        "val_years": ",".join(str(y) for y in years),
        "core_cost": best_cost(base, last_col),
        "test_cost": best_cost(test, last_col),
        "cost_change": change,
        "core_onset_recall": base_pick["onset_recall"],
        "test_onset_recall": test_pick["onset_recall"],
        # Overall scores of the test version at its chosen threshold (for comparison with other models)
        "test_accuracy": test_pick["accuracy"],
        "test_balanced_accuracy": test_pick["balanced_accuracy"],
        "test_macro_f1": test_pick["macro_f1"],
        "core_accuracy": base_pick["accuracy"],
        "core_balanced_accuracy": base_pick["balanced_accuracy"],
        "core_macro_f1": base_pick["macro_f1"],
        "helps": change < -noise,
    }

# First year with data for each environmental group
START_YEAR = {"chlorophyll": 2018, "wind": 2019}

def run_feature_tests(data, core, candidates, last_col, noise, **forest_args):
    tests = [("drop " + f, [c for c in core if c != f], None)
             for f in ["weeks_since_bloom", "reg_last_class", "site_last_class", "last_class"] if f in core]
    tests += [("add " + group, core + cols, START_YEAR.get(group)) for group, cols in candidates.items()]
    return pd.DataFrame([feature_test(data, core, last_col, name, feats, start, noise, **forest_args)
                         for name, feats, start in tests])

site_tests = run_feature_tests(df, features, feature_sets["site_candidates"], "site_last_class", SITE_NOISE, **SITE_ARGS)
site_tests

,test,trained_from,val_years,core_cost,test_cost,cost_change,core_onset_recall,test_onset_recall,test_accuracy,test_balanced_accuracy,test_macro_f1,core_accuracy,core_balanced_accuracy,core_macro_f1,helps
0,drop weeks_since_bloom,2015,"2019,2020,2021,2024",934,910,-24,0.519,0.467,0.890,0.623,0.552,0.874,0.619,0.532,False
1,drop reg_last_class,2015,"2019,2020,2021,2024",934,903,-31,0.519,0.550,0.881,0.636,0.549,0.874,0.619,0.532,False
2,drop site_last_class,2015,"2019,2020,2021,2024",934,930,-4,0.519,0.591,0.864,0.621,0.521,0.874,0.619,0.532,False
3,add temperature,2015,"2019,2020,2021,2024",934,923,-11,0.519,0.567,0.875,0.633,0.544,0.874,0.619,0.532,False
4,add chlorophyll,2018,"2019,2020,2021,2024",1002,996,-6,0.399,0.416,0.881,0.597,0.546,0.887,0.596,0.545,False
5,add wind,2019,"2020,2021,2024",667,664,-3,0.591,0.577,0.858,0.573,0.464,0.848,0.543,0.400,False
6,add rain_optional,2015,"2019,2020,2021,2024",934,941,7,0.519,0.543,0.871,0.631,0.534,0.874,0.619,0.532,False


### Final site feature set

Apply every change that helped, then check the combination still beats the core features (changes can overlap, so they don't simply add up). `site_last_class` is never dropped: it is the persistence information the client would expect the model to use.

In [18]:
def apply_changes(core, candidates, tests):
    feats = list(core)
    for name in tests.loc[tests["helps"], "test"]:
        action, target = name.split(" ", 1)
        if action == "drop" and target not in ("site_last_class", "last_class"):
            feats.remove(target)
        elif action == "add" and target not in START_YEAR:   # groups with short records would cut the training years
            feats += candidates[target]
    return feats

site_final_features = apply_changes(features, feature_sets["site_candidates"], site_tests)
site_final_preds = rolling_predictions(df, site_final_features, "site_last_class", **SITE_ARGS)

print("Final site features:", site_final_features)
print("Core lowest cost: ", best_cost(site_preds, "site_last_class"))
print("Final lowest cost:", best_cost(site_final_preds, "site_last_class"))

# Keep the combination only if it beats the core features by more than the noise
if best_cost(site_final_preds, "site_last_class") >= best_cost(site_preds, "site_last_class") - SITE_NOISE:
    site_final_features, site_final_preds = features, site_preds
    print("Combination did not clearly help, so the core features are kept")

site_choice = pick_threshold(site_final_preds, "site_last_class")
SITE_THRESHOLD = site_choice["threshold"]
print("Chosen threshold:", SITE_THRESHOLD)
pd.DataFrame([site_choice])

Final site features: ['site_last_class', 'site_last_log_cells', 'site_weeks_since_obs', 'nbr_class', 'reg_last_class', 'reg_class_max_4wk', 'reg_n_samples', 'weeks_since_bloom', 'pos_weeks_12wk', 'class_run_weeks', 'season_sin', 'season_cos']
Core lowest cost:  934
Final lowest cost: 934
Combination did not clearly help, so the core features are kept


Chosen threshold: 0.6


,threshold,accuracy,balanced_accuracy,macro_f1,onset_recall,onsets_caught,misses,bloom_misses,false_alarms,cost_2to1,cost_3to1
11,0.6,0.874,0.619,0.532,0.519,151 / 291,151,33,520,822,973


### Site-level reliability across sites

The client wants site-level prediction to stay secondary until it works across sites. This table splits the rolling-validation results of the final site model by part of the coast (thirds by site latitude). (Every site in the table is marked offshore, so an inshore/offshore split isn't possible.)

In [19]:
site_final_preds["pred_class"] = to_class(site_final_preds, SITE_THRESHOLD)
site_final_preds["alert"] = site_final_preds["pred_class"] > 0

# Split the coast into thirds by each site's latitude
site_lat = site_final_preds.groupby("site_key")["latitude"].first()
band = pd.qcut(site_lat, 3, labels=["south", "central", "north"])
site_final_preds["coast_band"] = site_final_preds["site_key"].map(band)

rows = []
for name, g in site_final_preds.groupby("coast_band", observed=True):
    rows.append({"coast_band": name, "sites": g["site_key"].nunique(), "site_weeks": len(g),
                 **early_warning_report(g["y_class"], g["pred_class"], g["site_last_class"])})
pd.DataFrame(rows)

,coast_band,sites,site_weeks,accuracy,balanced_accuracy,macro_f1,onset_recall,onsets_caught,misses,bloom_misses,false_alarms,cost_2to1,cost_3to1
0,south,46,2308,0.872,0.613,0.566,0.488,62 / 127,73,25,176,322,395
1,central,45,1960,0.860,0.579,0.493,0.581,54 / 93,39,8,182,260,299
2,north,45,2178,0.888,0.665,0.498,0.493,35 / 71,39,0,162,240,279


In [20]:
# Onset recall at each site that had at least 5 onsets
per_site = []
for key, g in site_final_preds.groupby("site_key"):
    onset = (g["site_last_class"] == 0) & (g["y_class"] > 0)
    if onset.sum() >= 5:
        per_site.append({"site_key": key, "onsets": int(onset.sum()), "onset_recall": g.loc[onset, "alert"].mean()})
per_site = pd.DataFrame(per_site)

print("Sites with 5+ onsets:", len(per_site), "of", site_final_preds["site_key"].nunique())
print(per_site["onset_recall"].describe().round(2))

Sites with 5+ onsets: 21 of 136
count    21.00
mean      0.43
std       0.15
min       0.17
25%       0.36
50%       0.40
75%       0.50
max       0.80
Name: onset_recall, dtype: float64


# Regional model (main product)

One row per week for the whole focus coast (`model_table_weekly_fe_reduced`, `regional_core` features). Three ways to make the regional forecast are compared:

1. **Baseline**: next week = last week's regional class
2. **Regional forest**: trained directly on the weekly table
3. **Site forest, combined**: run the site forest, then take the **highest site probability** each week as the regional probability

Regional numbers are small: across the four validation years there are only 14 regional onsets, so a difference of one or two weeks is not meaningful.

In [21]:
reg_df = pd.read_parquet("../data/final/model_table_weekly_fe_reduced.parquet")
reg_df = reg_df[reg_df["y_class"].notna()].copy()
reg_df["y_class"] = reg_df["y_class"].astype(int)
reg_features = feature_sets["regional_core"]

print(pd.crosstab(reg_df["target_year"], reg_df["y_class"]))

y_class       0   1   2
target_year            
2015         36   3  12
2016         16  14  22
2017         24   8  20
2018         10   5  38
2019         38   3  11
2020         42  10   0
2021          2  12  13
2022          4   0   4
2023         13   4   0
2024         38   5   9
2025         45   1   6
2026         18   0   0


In [22]:
REG_ARGS = dict(class_weight="balanced", leaf=10)   # fewer rows here, so smaller leaves

reg_val = reg_df[reg_df["target_year"].isin(VAL_YEARS)]
reg_baseline = early_warning_report(reg_val["y_class"], reg_val["last_class"].astype(int), reg_val["last_class"])

# 2. Regional forest
reg_preds = rolling_predictions(reg_df, reg_features, "last_class", **REG_ARGS)

# 3. Final site forest combined into one weekly probability (highest site probability that week)
site_max = site_final_preds.groupby("week_start")[["p_alert", "p_bloom"]].max()
combined = reg_val[["week_start", "target_year", "y_class", "last_class"]].merge(site_max, on="week_start", how="left")
combined[["p_alert", "p_bloom"]] = combined[["p_alert", "p_bloom"]].fillna(0)   # weeks with no site prediction: no alert

regional_compare = pd.DataFrame([
    {"model": "Baseline: next week = last week", "threshold": None, **reg_baseline},
    {"model": "Regional forest", **pick_threshold(reg_preds, "last_class").to_dict()},
    {"model": "Site forest, combined", **pick_threshold(combined, "last_class").to_dict()},
])
regional_compare

,model,threshold,accuracy,balanced_accuracy,macro_f1,onset_recall,onsets_caught,misses,bloom_misses,false_alarms,cost_2to1,cost_3to1
0,Baseline: next week = last week,NaN,0.792,0.694,0.695,0.000,0 / 14,14,2,13,41,55
1,Regional forest,0.40,0.732,0.705,0.678,0.429,6 / 14,8,1,29,45,53
2,"Site forest, combined",0.25,0.721,0.686,0.673,0.357,5 / 14,9,0,29,47,56


In [23]:
print("Regional forest: trade-off at each threshold")
threshold_sweep(reg_preds, "last_class")

Regional forest: trade-off at each threshold


,threshold,accuracy,balanced_accuracy,macro_f1,onset_recall,onsets_caught,misses,bloom_misses,false_alarms,cost_2to1,cost_3to1
0,0.05,0.273,0.530,0.345,1.000,14 / 14,0,0,120,120,120
1,0.10,0.295,0.541,0.369,1.000,14 / 14,0,0,116,116,116
2,0.15,0.344,0.566,0.417,1.000,14 / 14,0,0,107,107,107
3,0.20,0.464,0.619,0.517,0.929,13 / 14,1,0,84,86,87
4,0.25,0.525,0.633,0.557,0.786,11 / 14,3,0,71,77,80
5,0.30,0.639,0.683,0.629,0.714,10 / 14,4,0,49,57,61
6,0.35,0.678,0.678,0.643,0.429,6 / 14,8,1,39,55,63
7,0.40,0.732,0.705,0.678,0.429,6 / 14,8,1,29,45,53
8,0.45,0.738,0.675,0.663,0.143,2 / 14,12,1,24,48,60
9,0.50,0.754,0.675,0.668,0.071,1 / 14,13,1,20,46,59


In [24]:
print("Site forest combined: trade-off at each threshold")
threshold_sweep(combined, "last_class")

Site forest combined: trade-off at each threshold


,threshold,accuracy,balanced_accuracy,macro_f1,onset_recall,onsets_caught,misses,bloom_misses,false_alarms,cost_2to1,cost_3to1
0,0.05,0.328,0.553,0.411,0.929,13 / 14,1,0,109,111,112
1,0.10,0.426,0.594,0.496,0.857,12 / 14,2,0,90,94,96
2,0.15,0.579,0.664,0.601,0.786,11 / 14,3,0,61,67,70
3,0.20,0.667,0.683,0.649,0.571,8 / 14,6,0,42,54,60
4,0.25,0.721,0.686,0.673,0.357,5 / 14,9,0,29,47,56
5,0.30,0.738,0.686,0.679,0.214,3 / 14,11,1,25,47,58
6,0.35,0.776,0.706,0.706,0.143,2 / 14,12,2,18,42,54
7,0.40,0.787,0.686,0.700,0.143,2 / 14,16,3,13,45,61
8,0.45,0.803,0.686,0.707,0.143,2 / 14,17,3,9,43,60
9,0.50,0.765,0.608,0.631,0.000,0 / 14,24,3,9,57,81


## Feature tests (regional)

In [25]:
reg_seed_costs = [best_cost(rolling_predictions(reg_df, reg_features, "last_class", seed=seed, **REG_ARGS), "last_class")
                  for seed in [1, 2, 3, 42]]
REG_NOISE = 2 * (max(reg_seed_costs) - min(reg_seed_costs))
print("Lowest 3:1 cost by seed:", reg_seed_costs, "| A change has to beat:", REG_NOISE)

reg_tests = run_feature_tests(reg_df, reg_features, feature_sets["regional_candidates"], "last_class", REG_NOISE, **REG_ARGS)
reg_tests

Lowest 3:1 cost by seed: [np.int64(55), np.int64(53), np.int64(52), np.int64(53)] | A change has to beat: 6


,test,trained_from,val_years,core_cost,test_cost,cost_change,core_onset_recall,test_onset_recall,test_accuracy,test_balanced_accuracy,test_macro_f1,core_accuracy,core_balanced_accuracy,core_macro_f1,helps
0,drop weeks_since_bloom,2015,"2019,2020,2021,2024",53,51,-2,0.429,0.071,0.798,0.697,0.698,0.732,0.705,0.678,False
1,drop last_class,2015,"2019,2020,2021,2024",53,53,0,0.429,0.143,0.776,0.694,0.688,0.732,0.705,0.678,False
2,add temperature,2015,"2019,2020,2021,2024",53,50,-3,0.429,0.071,0.803,0.700,0.702,0.732,0.705,0.678,False
3,add chlorophyll,2018,"2019,2020,2021,2024",68,77,9,0.357,0.429,0.623,0.608,0.568,0.683,0.630,0.619,False
4,add wind,2019,"2020,2021,2024",50,47,-3,0.200,0.200,0.718,0.648,0.614,0.710,0.633,0.607,False
5,add nutrients_bay,2015,"2019,2020,2021,2024",53,52,-1,0.429,0.214,0.760,0.694,0.682,0.732,0.705,0.678,False
6,add rain_optional,2015,"2019,2020,2021,2024",53,52,-1,0.429,0.071,0.787,0.691,0.690,0.732,0.705,0.678,False


## What the final site model relies on (cost-based permutation importance)

Same idea as Part 1, but the score is now the **lowest 3:1 cost**, added up over all four validation years. The threshold is picked again after each scramble. Otherwise a scrambled feature can look "helpful" just because it changes how many alerts are raised.

Importance = how much the cost goes **up** when a feature is scrambled, so positive means the forest relies on it.

In [26]:
def permutation_cost(data, feats, last_col, n_repeats=5, **forest_args):
    rng = np.random.default_rng(42)
    folds = []
    for year in VAL_YEARS:
        tr = data[data["target_year"] < year]
        va = data[data["target_year"] == year]
        folds.append((make_forest(**forest_args).fit(tr[feats], tr["y_class"]), va))

    def pooled_cost(shuffle=None):
        pieces = []
        for forest, va in folds:
            X = va[feats].copy()
            if shuffle:
                X[shuffle] = rng.permutation(X[shuffle].values)
            proba = forest.predict_proba(X)
            out = va[["y_class", last_col]].copy()
            out["p_alert"] = 1 - proba[:, list(forest.classes_).index(0)]
            out["p_bloom"] = proba[:, list(forest.classes_).index(2)]
            pieces.append(out)
        return best_cost(pd.concat(pieces), last_col)

    base = pooled_cost()
    rows = []
    for f in feats:
        increases = [pooled_cost(f) - base for _ in range(n_repeats)]
        rows.append({"feature": f, "cost_increase": np.mean(increases), "spread": np.std(increases)})
    return pd.DataFrame(rows).sort_values("cost_increase", ascending=False)

site_importance = permutation_cost(df, site_final_features, "site_last_class", **SITE_ARGS)
print(site_importance.round(1).to_string(index=False))

             feature  cost_increase  spread
           nbr_class          183.6    20.6
 site_last_log_cells           37.8    13.8
   weeks_since_bloom           14.8    11.9
      reg_last_class            2.6    13.0
site_weeks_since_obs           -1.8     3.5
     site_last_class           -2.4     5.9
   reg_class_max_4wk           -4.0     3.9
          season_sin          -12.2    14.0
     class_run_weeks          -20.4     2.9
      pos_weeks_12wk          -21.4     2.6
       reg_n_samples          -24.0     4.6
          season_cos          -28.4     6.1


In [27]:
reg_importance = permutation_cost(reg_df, reg_features, "last_class", **REG_ARGS)
print(reg_importance.round(1).to_string(index=False))

            feature  cost_increase  spread
         season_cos            3.0     1.9
    last_class_prev            1.6     0.8
     last_log_cells            1.2     4.1
         season_sin            0.6     0.5
      class_max_4wk            0.4     0.5
    weeks_since_obs            0.0     0.0
     pos_weeks_12wk            0.0     1.4
  weeks_since_bloom           -0.2     1.7
   n_habsos_samples           -0.6     0.5
    class_run_weeks           -1.2     0.4
last_log_cells_chg1           -1.6     1.4
         last_class           -3.8     2.5


# Final check on 2025 (test year, used once)

The chosen settings and thresholds are fixed above. Each model is retrained on every year up to 2024 and scored once on 2025. Nothing gets changed after looking at these numbers.

In [28]:
def final_test(data, feats, last_col, **forest_args):
    """Train on every year up to 2024, predict 2025 (same columns as rolling_predictions)."""
    return rolling_predictions(data[data["target_year"] <= 2025], feats, last_col, val_years=[2025], **forest_args)

REG_THRESHOLD = pick_threshold(reg_preds, "last_class")["threshold"]
COMBINED_THRESHOLD = pick_threshold(combined, "last_class")["threshold"]

# Site level
site_te = final_test(df, site_final_features, "site_last_class", **SITE_ARGS)
# Regional forest
reg_te = final_test(reg_df, reg_features, "last_class", **REG_ARGS)
# Site forest combined into a weekly regional probability
comb_te = reg_te[["week_start", "y_class", "last_class"]].merge(
    site_te.groupby("week_start")[["p_alert", "p_bloom"]].max(), on="week_start", how="left").fillna(0)

final = pd.DataFrame({
    "Site: baseline": early_warning_report(site_te["y_class"], site_te["site_last_class"].fillna(0).astype(int), site_te["site_last_class"]),
    f"Site: forest (t={SITE_THRESHOLD})": early_warning_report(site_te["y_class"], to_class(site_te, SITE_THRESHOLD), site_te["site_last_class"]),
    "Regional: baseline": early_warning_report(reg_te["y_class"], reg_te["last_class"].astype(int), reg_te["last_class"]),
    f"Regional: forest (t={REG_THRESHOLD})": early_warning_report(reg_te["y_class"], to_class(reg_te, REG_THRESHOLD), reg_te["last_class"]),
    f"Regional: site forest combined (t={COMBINED_THRESHOLD})": early_warning_report(comb_te["y_class"], to_class(comb_te, COMBINED_THRESHOLD), comb_te["last_class"]),
}).T
final

,accuracy,balanced_accuracy,macro_f1,onset_recall,onsets_caught,misses,bloom_misses,false_alarms,cost_2to1,cost_3to1
Site: baseline,0.907,0.499,0.463,0.0,0 / 33,33,11,81,147,180
Site: forest (t=0.6),0.869,0.592,0.444,0.848,28 / 33,5,3,156,166,171
Regional: baseline,0.962,0.659,0.637,NaN,0 / 0,0,0,1,1,1
Regional: forest (t=0.4),0.904,0.637,0.626,NaN,0 / 0,0,0,4,4,4
Regional: site forest combined (t=0.25),0.827,0.463,0.514,NaN,0 / 0,0,0,5,5,5
